In [12]:
import os
import urllib.request
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [13]:
# 1. CARGA DEL DATASET EN KAGGLE
# ==========================================
input_path = '/kaggle/input/'
data_file = None

for root, dirs, files in os.walk(input_path):
    for file in files:
        if file.endswith(('.csv', '.tsv', '.txt')) and data_file is None:
            data_file = os.path.join(root, file)

if not data_file:
    print("--> Descargando Restaurant_Reviews.tsv...")
    data_file = "Restaurant_Reviews.tsv"
    url = "https://raw.githubusercontent.com/zapata-diego/Restaurant-Reviews-Dataset/main/Restaurant_Reviews.tsv"
    urllib.request.urlretrieve(url, data_file)

print(f"--> Cargando archivo: {data_file}")
try:
    df = pd.read_csv(data_file, sep='\t')
except Exception:
    df = pd.read_csv(data_file)

print(f"Dimensiones iniciales: {df.shape}")

--> Cargando archivo: /kaggle/input/datasets/d4rklucif3r/restaurant-reviews/Restaurant_Reviews.tsv
Dimensiones iniciales: (1000, 2)


In [14]:
# 2. LIMPIEZA Y CORRECCIÓN DE ETIQUETAS
# ==========================================
text_col = 'Review' if 'Review' in df.columns else df.columns[0]
rating_col = 'Liked' if 'Liked' in df.columns else df.columns[1]

df = df.dropna(subset=[text_col, rating_col]).copy()

# Mantiene 0 (Negativo) y 1 (Positivo) sin alterarlos
df['label'] = df[rating_col].astype(int)
df = df.rename(columns={text_col: 'text'})[['text', 'label']]
df['text'] = df['text'].astype(str)

print("--> Conteo real de clases:")
print(df['label'].value_counts().to_dict())

--> Conteo real de clases:
{1: 500, 0: 500}


In [15]:
# 3. PARTICIONES (80 / 10 / 10)
# ==========================================
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df['label'])

print(f"Particiones -> Entrenar: {len(train_df)} | Validación: {len(val_df)} | Prueba: {len(test_df)}")

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

Particiones -> Entrenar: 800 | Validación: 100 | Prueba: 100


In [16]:
# 4. TOKENIZACIÓN CON BETO
# ==========================================
model_checkpoint = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding="max_length")

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
# 5. BASELINE PIPELINE
# ==========================================
print("\n--> 1. Calculando métricas del Baseline...")
baseline_pipe = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

test_texts = raw_datasets['test']['text']
test_labels = raw_datasets['test']['label']

baseline_preds = []
for text in test_texts:
    res = baseline_pipe(text[:512])[0]['label']
    baseline_preds.append(1 if res == 'POS' else 0)

base_acc = accuracy_score(test_labels, baseline_preds)
base_f1 = f1_score(test_labels, baseline_preds, average='weighted')

print(f"   Baseline Accuracy: {base_acc:.4f}")
print(f"   Baseline F1-Score: {base_f1:.4f}\n")


--> 1. Calculando métricas del Baseline...


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

   Baseline Accuracy: 0.8700
   Baseline F1-Score: 0.8700



In [18]:
# 6. CONFIGURACIÓN DE PYTORCH
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> Dispositivo de entrenamiento: {device}")

tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(tokenized_datasets["train"], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=16)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=16)

num_labels = len(set(raw_datasets['train']['label']))
print(f"--> Clases detectadas para clasificación: {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=num_labels
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

--> Dispositivo de entrenamiento: cpu
--> Clases detectadas para clasificación: 2


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

In [19]:
# 7. BUCLE DE ENTRENAMIENTO NATIVO (3 ÉPOCAS)
# ==========================================
epochs = 3
print(f"--> Iniciando Fine-Tuning de BETO ({epochs} épocas)...")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device).long()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Evaluación en validación por época
    model.eval()
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).long()
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="weighted")
    
    print(f"Época {epoch + 1}/{epochs} | Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

--> Iniciando Fine-Tuning de BETO (3 épocas)...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Época 1/3 | Loss: 0.6476 | Val Acc: 0.7700 | Val F1: 0.7681
Época 2/3 | Loss: 0.4177 | Val Acc: 0.8100 | Val F1: 0.8098
Época 3/3 | Loss: 0.2598 | Val Acc: 0.7300 | Val F1: 0.7238


In [20]:
# 8. EVALUACIÓN FINAL EN CONJUNTO DE PRUEBA
# ==========================================
print("\n--> Evaluando modelo fine-tuned en Test...")
model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device).long()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(labels.cpu().numpy())

final_acc = accuracy_score(test_targets, test_preds)
final_f1 = f1_score(test_targets, test_preds, average="weighted")

print(f"\n==========================================")
print(f"RESULTADOS COMPARATIVOS FINALES")
print(f"==========================================")
print(f"Baseline        -> Accuracy: {base_acc:.4f} | F1-Score: {base_f1:.4f}")
print(f"BETO Fine-Tuned -> Accuracy: {final_acc:.4f} | F1-Score: {final_f1:.4f}")
print(f"==========================================")


--> Evaluando modelo fine-tuned en Test...

RESULTADOS COMPARATIVOS FINALES
Baseline        -> Accuracy: 0.8700 | F1-Score: 0.8700
BETO Fine-Tuned -> Accuracy: 0.7300 | F1-Score: 0.7149
